In [10]:
# Handle to the workspace
from azure.ai.ml import MLClient

# Authentication package
from azure.identity import DefaultAzureCredential

import mltable

from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import json

In [7]:
# Set up Azure ML workspace details
subscription_id = "08539d0e-620d-424c-aea8-56853ed8fe3e"
resource_group = "dspro2"
workspace_name = "dspro2ml"

# Connect to Azure ML workspace
ml_client = MLClient(
    DefaultAzureCredential(), 
    subscription_id, 
    resource_group, 
    workspace_name
)

# Define the path to your Delta Lake table in Azure Data Lake Storage
delta_table_path = "abfss://gold@dspro2deltalake.dfs.core.windows.net/bostonhousing/"

# Create the MLTable YAML definition for the Delta Lake table
mltable_yaml_content = f"""
paths:
  - file: {delta_table_path}
transformations:
  - read_parquet: {{}}
type: mltable
"""

# Save the MLTable YAML file locally
os.makedirs("data", exist_ok=True)
with open("data/MLTable", "w") as f:
    f.write(mltable_yaml_content)

# Register the Delta Lake table as a dataset in Azure ML Studio
dataset_name = "bost_gold"
dataset_version = "1"

data_asset = Data(
    path="./data",  # Path to the folder containing the MLTable file
    type=AssetTypes.MLTABLE,
    description="Delta Lake table registered as an MLTable dataset",
    name=dataset_name,
    version=dataset_version,
)

ml_client.data.create_or_update(data_asset)

print(f"Dataset '{dataset_name}' version '{dataset_version}' registered successfully.")


Uploading data (0.0 MBs): 100%|██████████| 127/127 [00:00<00:00, 9593.28it/s]




Dataset 'stats_gold' version '1' registered successfully.


In [ ]:
    subscription_id = '08539d0e-620d-424c-aea8-56853ed8fe3e'
    resource_group = 'dspro2'
    workspace_name = 'dspro2ml'

    workspace = Workspace(subscription_id, resource_group, workspace_name)
    dataset = Dataset.get_by_name(workspace, name=args.data)
    pddf_bh = dataset.to_pandas_dataframe()